<a href="https://colab.research.google.com/github/ZeroFiles/AML-Final-Ortiz-Larry/blob/main/notebooks/Tratamiento_de_Datos_Sinteticos_DataSet_CTGAN_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/ZeroFiles/AML-Final-Ortiz-Larry/blob/main/notebooks/Tratamiento_de_Datos_Sinteticos_DataSet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tratamiento de datos + generación sintética controlada con CTGAN

Este notebook prepara una base agregada por local-hora y, cuando el histórico real es insuficiente, permite generar datos sintéticos condicionados para cubrir combinaciones temporales faltantes.

Criterio metodológico:
- Los registros reales se conservan como fuente principal.
- Los registros sintéticos se marcan con `origen_dato = "sintetico_ctgan"`.
- Las variables temporales dependientes de continuidad, como lags y rolling, se recalculan después de unir real + sintético.
- La variable `flota_requerida_estimada` no se incluye como feature para evitar fuga de información.


In [4]:
# =====================================================
# 00) MONTAJE DE GOOGLE DRIVE
# =====================================================
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/AML_Final_Project/'

print("✅ Google Drive montado")
print(f"📂 Ruta base del proyecto: {BASE_PATH}")


Mounted at /content/drive
✅ Google Drive montado
📂 Ruta base del proyecto: /content/drive/MyDrive/AML_Final_Project/


In [5]:
# =====================================================
# 01) LIBRERÍAS
# =====================================================

# En Colab, descomentar si SDV no está instalado
!pip install -q sdv pyarrow openpyxl

import pandas as pd
import numpy as np

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from sdv.sampling import DataFrameCondition
from sdv.evaluation.single_table import evaluate_quality

pd.set_option("display.max_columns", 100)

print("✅ Librerías cargadas")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.9/206.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 6.1 MB/s eta 0:00:00
✅ Librerías cargadas


In [22]:
# =====================================================
# 02) PARÁMETROS
# =====================================================

INPUT_FILE = BASE_PATH + "datasetPrevio.xlsx"

OUTPUT_FILE_REAL = BASE_PATH + "dataset_model_real.parquet"
OUTPUT_FILE_SYNTH = BASE_PATH + "dataset_sintetico_ctgan.parquet"
OUTPUT_FILE_AUGMENTED = BASE_PATH + "dataset_model_real_mas_sintetico.parquet"

TARGET_START = "2022-01-01"
TARGET_END = "2025-12-31"

GENERAR_GRILLA_EXTENDIDA = True

# Si quieres alinear horas a Perú
USE_LIMA_TZ = True
LIMA_TZ = "America/Lima"

# Frecuencia horaria
FREQ = "h"

# Filtrado por horas operativas detectadas por local
# FILTER_OPERATING_HOURS = True
# OPER_HOUR_MIN_POS_RATE = 0.02
FILTER_OPERATING_HOURS = False

# Generación sintética
USE_SYNTHETIC_DATA = True
CTGAN_EPOCHS = 300

# Para ejecución rápida de prueba, limitar las filas faltantes a sintetizar.
# Usar None para generar todas las combinaciones faltantes.
MAX_SYNTHETIC_ROWS = None

# Lags / rolling
LAGS = [1, 2, 3, 24, 168]
ROLL_WINDOWS = [6, 12, 24, 168]

# Columnas base que CTGAN aprenderá.
# No incluir lags, rolling, ts_hour ni flota_requerida_estimada.
CTGAN_COLUMNS = [
    "local",
    "pedidos",
    "km_mean",
    "t_ret_mean",
    "t_ret_p75",
    "hora",
    "dow",
    "is_weekend"
]

CONDITION_COLUMNS = [
    "local",
    #"hora",
    #"dow",
    #"is_weekend"
]

print("✅ Parámetros configurados")


✅ Parámetros configurados


In [23]:
# =====================================================
# 03) CARGA
# =====================================================

# df = pd.read_excel(INPUT_FILE)

# print("Shape original:", df.shape)
# display(df.head())


# =====================================================
# FILTRO DE ORGANIZACIÓN Y LOCAL OBJETIVO
# =====================================================

df = pd.read_excel(INPUT_FILE)

ORG_OBJETIVO = "Don Tito"
LOCAL_OBJETIVO = "DT San Borja"

df = df[
    (df["organizacion"].astype(str).str.strip() == ORG_OBJETIVO) &
    (df["local"].astype(str).str.strip() == LOCAL_OBJETIVO)
].copy()

print("Shape filtrado:", df.shape)
print("Organización:", ORG_OBJETIVO)
print("Local:", LOCAL_OBJETIVO)
display(df.head())


Shape filtrado: (1395, 11)
Organización: Don Tito
Local: DT San Borja


,order_id,organizacion,local,id_conductor,fecha_creacion_fv,KM_REAL,TIEMPO_ENTREGA_RETORNO,hora_inicial,fecha_creacion_date,real_tiempo_inicial,real_tiempo_final
5,680e6a189ad43db803196b04,Don Tito,DT San Borja,66d64f6ed5c810dbbb682573,2025-04-27 12:31:40.494,0.094300,8.0,12,2025-04-27,2025-04-27 12:35:10 UTC,2025-04-27 12:41:00 UTC
12,67f334bc4e5b3161664cb409,Don Tito,DT San Borja,66d6552dd5c810dbbb6825d3,2025-04-06 21:09:34.315,0.698827,18.0,21,2025-04-06,2025-04-06 21:16:14 UTC,2025-04-06 21:27:13 UTC
16,67f2bae14e5b3161664c912b,Don Tito,DT San Borja,67d606067a1cd1a07c48d312,2025-04-06 12:33:14.391,0.953038,28.0,12,2025-04-06,2025-04-06 12:34:48 UTC,2025-04-06 12:53:21 UTC
17,67fc04d24e5b3161664d1ea5,Don Tito,DT San Borja,67f2a0f84e5b3161664c8dac,2025-04-13 13:35:04.470,0.188728,12.0,13,2025-04-13,2025-04-13 13:55:48 UTC,2025-04-13 14:03:51 UTC
18,680e78259ad43db803197267,Don Tito,DT San Borja,67d606067a1cd1a07c48d312,2025-04-27 13:31:35.312,0.723813,28.0,13,2025-04-27,2025-04-27 14:05:10 UTC,2025-04-27 14:19:20 UTC


In [24]:
# =====================================================
# 04) TIPOS Y LIMPIEZA
# =====================================================

df["fecha_creacion_fv"] = pd.to_datetime(
    df["fecha_creacion_fv"],
    errors="coerce",
    utc=True
)

for col in ["KM_REAL", "TIEMPO_ENTREGA_RETORNO"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["fecha_creacion_fv", "local"]).copy()

# Convertir a Lima si aplica
if USE_LIMA_TZ:
    df["fecha_creacion_fv"] = df["fecha_creacion_fv"].dt.tz_convert(LIMA_TZ)

# Bucket horario
df["ts_hour"] = df["fecha_creacion_fv"].dt.floor("h")
df["hora"] = df["ts_hour"].dt.hour

print("Shape luego de limpieza:", df.shape)
print("Rango real observado:", df["ts_hour"].min(), "->", df["ts_hour"].max())
print("Locales:", df["local"].nunique())


Shape luego de limpieza: (1395, 13)
Rango real observado: 2025-02-14 06:00:00-05:00 -> 2025-05-04 16:00:00-05:00
Locales: 1


In [25]:
# =====================================================
# 05) AGREGACIÓN BASE OBSERVADA (local-hora)
# =====================================================

base_observada = (
    df.groupby(["local", "ts_hour"], as_index=False)
      .agg(
          pedidos=("order_id", "count"),
          km_mean=("KM_REAL", "mean"),
          t_ret_mean=("TIEMPO_ENTREGA_RETORNO", "mean"),
          t_ret_p75=("TIEMPO_ENTREGA_RETORNO",
                     lambda x: np.nanpercentile(x, 75) if np.isfinite(x).any() else np.nan)
      )
)

base_observada["origen_dato"] = "real"

print("Shape base observada:", base_observada.shape)
display(base_observada.head())


Shape base observada: (78, 7)


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,origen_dato
0,DT San Borja,2025-02-14 06:00:00-05:00,1,0.011145,4.0,4.0,real
1,DT San Borja,2025-02-14 08:00:00-05:00,1,0.259452,4.0,4.0,real
2,DT San Borja,2025-02-14 10:00:00-05:00,1,0.005896,NaN,NaN,real
3,DT San Borja,2025-02-14 12:00:00-05:00,1,0.005401,4.0,4.0,real
4,DT San Borja,2025-02-14 13:00:00-05:00,2,0.001188,5.0,5.5,real


In [26]:
# =====================================================
# 06) GRILLA CALENDARIO OBJETIVO POR LOCAL
# =====================================================

def build_calendar_grid_per_local(base_df: pd.DataFrame) -> pd.DataFrame:
    out = []

    for loc, b in base_df.groupby("local", sort=False):

        if GENERAR_GRILLA_EXTENDIDA:
            if USE_LIMA_TZ:
                min_ts = pd.Timestamp(TARGET_START, tz=LIMA_TZ)
                max_ts = pd.Timestamp(TARGET_END, tz=LIMA_TZ) + pd.Timedelta(hours=23)
            else:
                min_ts = pd.Timestamp(TARGET_START)
                max_ts = pd.Timestamp(TARGET_END) + pd.Timedelta(hours=23)
        else:
            min_ts = b["ts_hour"].min().floor("D")
            max_ts = b["ts_hour"].max().ceil("D") - pd.Timedelta(hours=1)

        hours = pd.date_range(
            start=min_ts,
            end=max_ts,
            freq=FREQ
        )

        idx = pd.MultiIndex.from_product(
            [[loc], hours],
            names=["local", "ts_hour"]
        )

        out.append(idx.to_frame(index=False))

    return pd.concat(out, ignore_index=True)


grilla = build_calendar_grid_per_local(base_observada)

base_keys = base_observada[["local", "ts_hour"]].drop_duplicates()

grilla = grilla.merge(
    base_keys.assign(existe_real=1),
    on=["local", "ts_hour"],
    how="left"
)

grilla["existe_real"] = grilla["existe_real"].fillna(0).astype(int)

print("Total combinaciones local-hora en grilla:", len(grilla))
print("Combinaciones reales observadas:", grilla["existe_real"].sum())
print("Combinaciones faltantes:", (grilla["existe_real"] == 0).sum())

display(grilla.head())


Total combinaciones local-hora en grilla: 35064
Combinaciones reales observadas: 78
Combinaciones faltantes: 34986


,local,ts_hour,existe_real
0,DT San Borja,2022-01-01 00:00:00-05:00,0
1,DT San Borja,2022-01-01 01:00:00-05:00,0
2,DT San Borja,2022-01-01 02:00:00-05:00,0
3,DT San Borja,2022-01-01 03:00:00-05:00,0
4,DT San Borja,2022-01-01 04:00:00-05:00,0


In [27]:
# =====================================================
# 07) VARIABLES CALENDARIO Y FILTRO DE HORAS OPERATIVAS
# =====================================================

grilla["hora"] = grilla["ts_hour"].dt.hour
grilla["dow"] = grilla["ts_hour"].dt.dayofweek
grilla["month"] = grilla["ts_hour"].dt.month
grilla["is_weekend"] = (grilla["dow"] >= 5).astype(int)

base_observada["hora"] = base_observada["ts_hour"].dt.hour
base_observada["dow"] = base_observada["ts_hour"].dt.dayofweek
base_observada["month"] = base_observada["ts_hour"].dt.month
base_observada["is_weekend"] = (base_observada["dow"] >= 5).astype(int)

# =====================================================
# FILTRO MANUAL DE HORAS OPERATIVAS
# =====================================================

OPERATING_HOURS_MANUAL = list(range(10, 24))

grilla = grilla[grilla["hora"].isin(OPERATING_HOURS_MANUAL)].copy()
base_observada = base_observada[base_observada["hora"].isin(OPERATING_HOURS_MANUAL)].copy()

if FILTER_OPERATING_HOURS:
    pos_rate = (
        base_observada.assign(pos=(base_observada["pedidos"] > 0).astype(int))
                      .groupby(["local", "hora"])["pos"]
                      .mean()
                      .reset_index()
                      .rename(columns={"pos": "pos_rate"})
    )

    grilla = grilla.merge(pos_rate, on=["local", "hora"], how="left")
    grilla["pos_rate"] = grilla["pos_rate"].fillna(0)

    grilla = grilla[grilla["pos_rate"] >= OPER_HOUR_MIN_POS_RATE].copy()
    grilla.drop(columns=["pos_rate"], inplace=True)

    horas_operativas = grilla[["local", "hora"]].drop_duplicates()

    base_observada = base_observada.merge(
        horas_operativas.assign(es_hora_operativa=1),
        on=["local", "hora"],
        how="left"
    )

    base_observada = base_observada[
        base_observada["es_hora_operativa"].fillna(0).eq(1)
    ].copy()

    base_observada.drop(columns=["es_hora_operativa"], inplace=True)

print("Grilla luego de filtro operativo:", grilla.shape)
print("Base observada luego de filtro operativo:", base_observada.shape)


Grilla luego de filtro operativo: (20454, 7)
Base observada luego de filtro operativo: (52, 11)


In [28]:
# =====================================================
# 08) GENERACIÓN SINTÉTICA CON CTGAN
# =====================================================

def preparar_datos_ctgan(base_real: pd.DataFrame) -> pd.DataFrame:
    train_ctgan = base_real[CTGAN_COLUMNS].copy()

    # Imputación robusta para variables operativas
    for col in ["km_mean", "t_ret_mean", "t_ret_p75"]:
        train_ctgan[col] = train_ctgan[col].fillna(train_ctgan[col].median())

    # SDV trabaja mejor si las categóricas están claramente definidas
    for col in CONDITION_COLUMNS:
        train_ctgan[col] = train_ctgan[col].astype(str)

    # Asegurar tipos numéricos
    for col in ["pedidos", "km_mean", "t_ret_mean", "t_ret_p75"]:
        train_ctgan[col] = pd.to_numeric(train_ctgan[col], errors="coerce")

    train_ctgan = train_ctgan.dropna().copy()
    train_ctgan["pedidos"] = train_ctgan["pedidos"].round().astype(int)

    return train_ctgan


if USE_SYNTHETIC_DATA:
    train_ctgan = preparar_datos_ctgan(base_observada)

    print("Shape train CTGAN:", train_ctgan.shape)
    display(train_ctgan.head())

    metadata = Metadata.detect_from_dataframe(
        data=train_ctgan,
        table_name="demanda_local_hora"
    )

    for col in CONDITION_COLUMNS:
        metadata.update_column(
            table_name="demanda_local_hora",
            column_name=col,
            sdtype="categorical"
        )

    for col in ["pedidos", "km_mean", "t_ret_mean", "t_ret_p75"]:
        metadata.update_column(
            table_name="demanda_local_hora",
            column_name=col,
            sdtype="numerical"
        )

    synthesizer = CTGANSynthesizer(
        metadata,
        epochs=CTGAN_EPOCHS,
        enforce_rounding=True,
        enforce_min_max_values=True,
        verbose=True
    )

    synthesizer.fit(train_ctgan)

    print("✅ CTGAN entrenado")
else:
    train_ctgan = None
    synthesizer = None
    print("⚠️ Generación sintética desactivada")


Shape train CTGAN: (52, 8)


,local,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,is_weekend
2,DT San Borja,1,0.005896,27.386364,33.5,10,4,0
3,DT San Borja,1,0.005401,4.000000,4.0,12,4,0
4,DT San Borja,2,0.001188,5.000000,5.5,13,4,0
5,DT San Borja,2,0.004708,2.000000,2.0,14,4,0
6,DT San Borja,2,0.001640,23.000000,24.5,15,4,0


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:138: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-00.36) | Discrim. (-00.03): 100%|██████████| 300/300 [00:23<00:00, 13.02it/s]

✅ CTGAN entrenado


In [29]:
import pandas as pd
import numpy as np

from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
from sdv.sampling import Condition # Changed from DataFrameCondition
from sdv.evaluation.single_table import evaluate_quality

# =====================================================
# 09) GENERAR DATA SINTÉTICA PARA COMBINACIONES FALTANTES
# =====================================================

# MAX_SYNTHETIC_ROWS = 3000

if USE_SYNTHETIC_DATA:
    faltantes = grilla[grilla["existe_real"] == 0].copy()

    if MAX_SYNTHETIC_ROWS is not None:
        faltantes = faltantes.head(MAX_SYNTHETIC_ROWS).copy()

    n_synth = len(faltantes)

    print("Filas faltantes candidatas a sintetizar:", len(grilla[grilla["existe_real"] == 0]))
    print("Filas faltantes a sintetizar:", n_synth)

    if n_synth > 0:

        # Generación libre, sin condiciones estrictas
        synthetic_data = synthesizer.sample(num_rows=n_synth)

        # Asegurar la misma cantidad de filas
        synthetic_data = synthetic_data.head(n_synth).copy()

        # Asignar local y calendario desde la grilla faltante
        synthetic_data["local"] = faltantes["local"].astype(str).values
        synthetic_data["ts_hour"] = faltantes["ts_hour"].values
        synthetic_data["hora"] = faltantes["hora"].astype(int).values
        synthetic_data["dow"] = faltantes["dow"].astype(int).values
        synthetic_data["month"] = faltantes["month"].astype(int).values
        synthetic_data["is_weekend"] = faltantes["is_weekend"].astype(int).values
        synthetic_data["origen_dato"] = "sintetico_ctgan"

        # Ordenar columnas
        synthetic_data = synthetic_data[
            [
                "local",
                "ts_hour",
                "pedidos",
                "km_mean",
                "t_ret_mean",
                "t_ret_p75",
                "hora",
                "dow",
                "month",
                "is_weekend",
                "origen_dato"
            ]
        ].copy()

        print("Shape data sintética:", synthetic_data.shape)
        display(synthetic_data.head())

    else:
        synthetic_data = pd.DataFrame()
        print("No existen combinaciones faltantes para sintetizar.")

else:
    synthetic_data = pd.DataFrame()

Filas faltantes candidatas a sintetizar: 20402
Filas faltantes a sintetizar: 20402
Shape data sintética: (20402, 11)


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,origen_dato
0,DT San Borja,2022-01-01 15:00:00,6,1.454070,32.747973,35.9,10,5,1,1,sintetico_ctgan
1,DT San Borja,2022-01-01 16:00:00,1,1.218012,18.782757,27.6,11,5,1,1,sintetico_ctgan
2,DT San Borja,2022-01-01 17:00:00,10,1.785659,32.361087,20.1,12,5,1,1,sintetico_ctgan
3,DT San Borja,2022-01-01 18:00:00,5,1.303569,2.000000,24.3,13,5,1,1,sintetico_ctgan
4,DT San Borja,2022-01-01 19:00:00,1,1.230435,27.001245,32.9,14,5,1,1,sintetico_ctgan


In [30]:
# =====================================================
# 10) POSTPROCESO Y REGLAS DE NEGOCIO SOBRE DATA SINTÉTICA
# =====================================================

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    # Corrección de tipos
    for col in CONDITION_COLUMNS:
        if col != "local":
            synthetic_data[col] = pd.to_numeric(
                synthetic_data[col],
                errors="coerce"
            ).round().astype("Int64")

    synthetic_data["local"] = synthetic_data["local"].astype(str)

    # Reglas de rango
    synthetic_data["pedidos"] = (
        pd.to_numeric(synthetic_data["pedidos"], errors="coerce")
        .fillna(0)
        .round()
        .clip(lower=0)
        .astype(int)
    )

    for col in ["km_mean", "t_ret_mean", "t_ret_p75"]:
        synthetic_data[col] = (
            pd.to_numeric(synthetic_data[col], errors="coerce")
            .fillna(train_ctgan[col].median())
            .clip(lower=0)
        )

    synthetic_data["hora"] = synthetic_data["hora"].clip(0, 23).astype(int)
    synthetic_data["dow"] = synthetic_data["dow"].clip(0, 6).astype(int)
    synthetic_data["month"] = synthetic_data["month"].clip(1, 12).astype(int)
    synthetic_data["is_weekend"] = synthetic_data["is_weekend"].clip(0, 1).astype(int)

    # Orden de columnas
    synthetic_data = synthetic_data[
        [
            "local",
            "ts_hour",
            "pedidos",
            "km_mean",
            "t_ret_mean",
            "t_ret_p75",
            "hora",
            "dow",
            "month",
            "is_weekend",
            "origen_dato"
        ]
    ].copy()

    print("✅ Postproceso sintético completado")
    display(synthetic_data.head())
else:
    print("No hay data sintética para postprocesar")


✅ Postproceso sintético completado


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,origen_dato
0,DT San Borja,2022-01-01 15:00:00,6,1.454070,32.747973,35.9,10,5,1,1,sintetico_ctgan
1,DT San Borja,2022-01-01 16:00:00,1,1.218012,18.782757,27.6,11,5,1,1,sintetico_ctgan
2,DT San Borja,2022-01-01 17:00:00,10,1.785659,32.361087,20.1,12,5,1,1,sintetico_ctgan
3,DT San Borja,2022-01-01 18:00:00,5,1.303569,2.000000,24.3,13,5,1,1,sintetico_ctgan
4,DT San Borja,2022-01-01 19:00:00,1,1.230435,27.001245,32.9,14,5,1,1,sintetico_ctgan


In [31]:
# =====================================================
# 11) UNIÓN REAL + SINTÉTICO
# =====================================================

cols_base = [
    "local",
    "ts_hour",
    "pedidos",
    "km_mean",
    "t_ret_mean",
    "t_ret_p75",
    "hora",
    "dow",
    "month",
    "is_weekend",
    "origen_dato"
]

base_real = base_observada[cols_base].copy()

# -----------------------------------------------------
# Normalizar ts_hour en base real
# -----------------------------------------------------
base_real["ts_hour"] = pd.to_datetime(
    base_real["ts_hour"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

# -----------------------------------------------------
# Normalizar ts_hour en data sintética
# -----------------------------------------------------
if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    synthetic_data = synthetic_data.copy()

    synthetic_data["ts_hour"] = pd.to_datetime(
        synthetic_data["ts_hour"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

    base_full = pd.concat(
        [base_real, synthetic_data[cols_base]],
        ignore_index=True
    )
else:
    base_full = base_real.copy()

# -----------------------------------------------------
# Normalización final de tipos
# -----------------------------------------------------
base_full["local"] = base_full["local"].astype(str)

for col in ["hora", "dow", "month", "is_weekend"]:
    base_full[col] = pd.to_numeric(base_full[col], errors="coerce").astype("Int64")

for col in ["pedidos", "km_mean", "t_ret_mean", "t_ret_p75"]:
    base_full[col] = pd.to_numeric(base_full[col], errors="coerce")

# pedidos debe ser entero no negativo
base_full["pedidos"] = (
    base_full["pedidos"]
    .round()
    .clip(lower=0)
    .astype("Int64")
)

# Eliminar filas con fecha inválida, si existieran
base_full = base_full.dropna(subset=["ts_hour"]).copy()

# Ordenar base consolidada
base_full = base_full.sort_values(["local", "ts_hour"]).reset_index(drop=True)

print("Shape base consolidada:", base_full.shape)
print(base_full["origen_dato"].value_counts())
print("\nTipo de ts_hour:", base_full["ts_hour"].dtype)

display(base_full.head())


Shape base consolidada: (20454, 11)
origen_dato
sintetico_ctgan    20402
real                  52
Name: count, dtype: int64

Tipo de ts_hour: datetime64[ns]


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,origen_dato
0,DT San Borja,2022-01-01 15:00:00,6,1.454070,32.747973,35.9,10,5,1,1,sintetico_ctgan
1,DT San Borja,2022-01-01 16:00:00,1,1.218012,18.782757,27.6,11,5,1,1,sintetico_ctgan
2,DT San Borja,2022-01-01 17:00:00,10,1.785659,32.361087,20.1,12,5,1,1,sintetico_ctgan
3,DT San Borja,2022-01-01 18:00:00,5,1.303569,2.000000,24.3,13,5,1,1,sintetico_ctgan
4,DT San Borja,2022-01-01 19:00:00,1,1.230435,27.001245,32.9,14,5,1,1,sintetico_ctgan


In [32]:
# =====================================================
# 12) FEATURES TEMPORALES
# =====================================================

base_full["hora"] = base_full["ts_hour"].dt.hour
base_full["dow"] = base_full["ts_hour"].dt.dayofweek
base_full["month"] = base_full["ts_hour"].dt.month
base_full["is_weekend"] = (base_full["dow"] >= 5).astype(int)

# Variables cíclicas
base_full["hora_sin"] = np.sin(2 * np.pi * base_full["hora"] / 24)
base_full["hora_cos"] = np.cos(2 * np.pi * base_full["hora"] / 24)
base_full["dow_sin"] = np.sin(2 * np.pi * base_full["dow"] / 7)
base_full["dow_cos"] = np.cos(2 * np.pi * base_full["dow"] / 7)

base_full = base_full.sort_values(["local", "ts_hour"]).reset_index(drop=True)

print("✅ Features temporales recalculadas")
display(base_full.head())


✅ Features temporales recalculadas


,local,ts_hour,pedidos,km_mean,t_ret_mean,t_ret_p75,hora,dow,month,is_weekend,origen_dato,hora_sin,hora_cos,dow_sin,dow_cos
0,DT San Borja,2022-01-01 15:00:00,6,1.454070,32.747973,35.9,15,5,1,1,sintetico_ctgan,-0.707107,-7.071068e-01,-0.974928,-0.222521
1,DT San Borja,2022-01-01 16:00:00,1,1.218012,18.782757,27.6,16,5,1,1,sintetico_ctgan,-0.866025,-5.000000e-01,-0.974928,-0.222521
2,DT San Borja,2022-01-01 17:00:00,10,1.785659,32.361087,20.1,17,5,1,1,sintetico_ctgan,-0.965926,-2.588190e-01,-0.974928,-0.222521
3,DT San Borja,2022-01-01 18:00:00,5,1.303569,2.000000,24.3,18,5,1,1,sintetico_ctgan,-1.000000,-1.836970e-16,-0.974928,-0.222521
4,DT San Borja,2022-01-01 19:00:00,1,1.230435,27.001245,32.9,19,5,1,1,sintetico_ctgan,-0.965926,2.588190e-01,-0.974928,-0.222521


In [33]:
# =====================================================
# 13) LAGS
# =====================================================

for lag in LAGS:
    base_full[f"pedidos_lag_{lag}h"] = (
        base_full.groupby("local")["pedidos"].shift(lag)
    )

print("✅ Lags calculados:", LAGS)


✅ Lags calculados: [1, 2, 3, 24, 168]


In [34]:
# =====================================================
# 14) ROLLING
# =====================================================

for w in ROLL_WINDOWS:
    shifted = base_full.groupby("local")["pedidos"].shift(1)

    base_full[f"pedidos_roll_mean_{w}h"] = (
        shifted.groupby(base_full["local"])
               .rolling(window=w, min_periods=max(2, w // 3))
               .mean()
               .reset_index(level=0, drop=True)
    )

    base_full[f"pedidos_roll_std_{w}h"] = (
        shifted.groupby(base_full["local"])
               .rolling(window=w, min_periods=max(2, w // 3))
               .std()
               .reset_index(level=0, drop=True)
    )

print("✅ Rolling calculados:", ROLL_WINDOWS)


✅ Rolling calculados: [6, 12, 24, 168]


In [35]:
# =====================================================
# 15) VALIDACIÓN BÁSICA DE DATA SINTÉTICA
# =====================================================

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    print("Distribución de pedidos reales observados:")
    display(base_real["pedidos"].describe())

    print("Distribución de pedidos sintéticos:")
    display(synthetic_data["pedidos"].describe())

    comparacion_hora = pd.DataFrame({
        "real": base_real.groupby("hora")["pedidos"].mean(),
        "sintetico": synthetic_data.groupby("hora")["pedidos"].mean()
    })

    print("Pedidos promedio por hora: real vs sintético")
    display(comparacion_hora)

    try:
        synthetic_eval = synthetic_data[CTGAN_COLUMNS].copy()
        for col in CONDITION_COLUMNS:
            synthetic_eval[col] = synthetic_eval[col].astype(str)

        quality_report = evaluate_quality(
            real_data=train_ctgan,
            synthetic_data=synthetic_eval,
            metadata=metadata
        )

        print("Score de calidad sintética:", quality_report.get_score())
    except Exception as e:
        print("No se pudo calcular evaluate_quality:", str(e))
else:
    print("No aplica validación sintética")


Distribución de pedidos reales observados:


,pedidos
count,52.000000
mean,9.288462
std,6.643105
min,1.000000
25%,4.000000
50%,8.500000
75%,11.250000
max,26.000000


Distribución de pedidos sintéticos:


,pedidos
count,20402.000000
mean,6.857906
std,7.242469
min,1.000000
25%,1.000000
50%,4.000000
75%,11.000000
max,26.000000


Pedidos promedio por hora: real vs sintético


,real,sintetico
hora,,
10,18.714286,6.848693
11,10.666667,6.557388
12,9.142857,6.801926
13,7.571429,6.797111
14,10.714286,7.068776
15,7.428571,6.877579
16,5.428571,6.839752
17,1.500000,6.855182
18,NaN,6.851472


Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 8/8 [00:00<00:00, 229.28it/s]|
Column Shapes Score: 67.41%

(2/2) Evaluating Column Pair Trends: |██████████| 28/28 [00:00<00:00, 111.87it/s]|
Column Pair Trends Score: 47.15%

Overall Score (Average): 57.28%

Score de calidad sintética: 0.5727839002796018


In [36]:
# =====================================================
# 16) FILTRADO FINAL MODEL-READY
# =====================================================

required = [
    f"pedidos_lag_{min(LAGS)}h",
    "pedidos_lag_24h",
    "pedidos_roll_mean_24h"
]

df_model = base_full.dropna(subset=required).copy()

print("✅ Shape model-ready:", df_model.shape)
print("Rango fechas:", df_model["ts_hour"].min(), "->", df_model["ts_hour"].max())
print("Locales:", df_model["local"].nunique())

print("\nDistribución pedidos model-ready:")
display(df_model["pedidos"].describe())

print("\nOrigen de datos model-ready:")
display(df_model["origen_dato"].value_counts())


✅ Shape model-ready: (20430, 28)
Rango fechas: 2022-01-03 01:00:00 -> 2026-01-01 04:00:00
Locales: 1

Distribución pedidos model-ready:


,pedidos
count,20430.0
mean,6.862115
std,7.23964
min,1.0
25%,1.0
50%,4.0
75%,11.0
max,26.0



Origen de datos model-ready:


,count
origen_dato,
sintetico_ctgan,20378
real,52


In [37]:
registros_x_mes_pivot = (
    base_full
    .assign(mes=base_full["ts_hour"].dt.to_period("M"))
    .pivot_table(
        index="mes",
        columns="origen_dato",
        values="ts_hour",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

registros_x_mes_pivot["total"] = registros_x_mes_pivot.drop(columns="mes").sum(axis=1)

display(registros_x_mes_pivot)

origen_dato,mes,real,sintetico_ctgan,total
0,2022-01,0,429,429
1,2022-02,0,392,392
2,2022-03,0,434,434
3,2022-04,0,420,420
4,2022-05,0,434,434
5,2022-06,0,420,420
6,2022-07,0,434,434
7,2022-08,0,434,434
8,2022-09,0,420,420
9,2022-10,0,434,434


In [38]:
# =====================================================
# 17) GUARDAR
# =====================================================

# Dataset model-ready con solo registros reales
df_model_real = df_model[df_model["origen_dato"] == "real"].copy()

df_model_real.to_parquet(OUTPUT_FILE_REAL, index=False)

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    synthetic_data.to_parquet(OUTPUT_FILE_SYNTH, index=False)
    df_model.to_parquet(OUTPUT_FILE_AUGMENTED, index=False)

print("✅ Dataset model-ready real guardado en:")
print(OUTPUT_FILE_REAL)

if USE_SYNTHETIC_DATA and len(synthetic_data) > 0:
    print("\n✅ Dataset sintético guardado en:")
    print(OUTPUT_FILE_SYNTH)

    print("\n✅ Dataset model-ready real + sintético guardado en:")
    print(OUTPUT_FILE_AUGMENTED)


✅ Dataset model-ready real guardado en:
/content/drive/MyDrive/AML_Final_Project/dataset_model_real.parquet

✅ Dataset sintético guardado en:
/content/drive/MyDrive/AML_Final_Project/dataset_sintetico_ctgan.parquet

✅ Dataset model-ready real + sintético guardado en:
/content/drive/MyDrive/AML_Final_Project/dataset_model_real_mas_sintetico.parquet
